# Getting LastFM Data
This notebook will ingest scrobbling data for a given username's LastFM account. 

## Setup
Load the API key and username from `lastfm.env`.

In [1]:
import os
import json
import time
import requests
import pandas as pd
from dotenv import load_dotenv
load_dotenv('lastfm.env')
API_KEY = os.environ['LASTFM_API_KEY']
USER = os.environ['LASTFM_USER']; print(f'User: {USER}')

URL = 'https://ws.audioscrobbler.com/2.0/'

User: sk3141


## Request parameters
`limit` is the number of scrobbles per page (max 200).

In [2]:
LIMIT = 200

params = {
    'method': 'user.getrecenttracks',
    'user': USER,
    'api_key': API_KEY,
    'format': 'json',
    'limit': LIMIT,
}

## Explore one response
The tracks are under `track`, and the pagination info is under `@attr`.

In [3]:
response = requests.get(URL, params=params, timeout=30)
print(response.status_code)

data = response.json()
#print(json.dumps(data, indent=2)[:3000])

200


In [4]:
print(data['recenttracks']['@attr'])
track = data['recenttracks']['track'][0]
print(track.keys())
track['artist']['#text']

{'perPage': '200', 'totalPages': '171', 'page': '1', 'user': 'sk3141', 'total': '34079'}
dict_keys(['artist', 'streamable', 'image', 'mbid', 'album', 'name', 'url', 'date'])


'Skizzy Mars'

## Parse a track
Flatten one scrobble into a dict. The track that is currently playing has no `date`, so it is skipped.

In [5]:
def parse_track(track):
    '''Flatten one scrobble from the API into a simple dict.
    Returns None for the now-playing track, which has no timestamp yet.
    '''
    if 'date' not in track:  # now-playing tracks have no 'date' key
        return None
    return {
        'artist': track['artist']['#text'],
        'album': track['album']['#text'],
        'track': track['name'],
        'uts': int(track['date']['uts']),  # Unix seconds (UTC), as an int
    }

In [6]:
tracks = data['recenttracks']['track']
print(parse_track(tracks[0]))

{'artist': 'Skizzy Mars', 'album': 'Alone Together', 'track': 'Silver Lining (feat. PJ & Rome Fortune)', 'uts': 1794543092}


## Fetch one page
`fetch_page` returns the parsed tracks for a single page. `raise_for_status()` raises an error if a request fails.

In [7]:
def fetch_page(page):
    '''Fetch one page of scrobbles and return them as a list of parsed dicts.'''
    page_params = {**params, 'page': page}  # copy so the shared params dict isn't mutated
    response = requests.get(URL, params=page_params, timeout=30)
    response.raise_for_status()  # raise on HTTP errors instead of returning bad data
    tracks = response.json()['recenttracks']['track']
    if isinstance(tracks, dict):  # the API returns a bare dict when a page has only one track
        tracks = [tracks]
    rows = [parse_track(track) for track in tracks]
    return [row for row in rows if row is not None]  # drop the now-playing track

In [8]:
test_rows = fetch_page(1)
print(len(test_rows))
test_rows[0]

200


{'artist': 'Skizzy Mars',
 'album': 'Alone Together',
 'track': 'Silver Lining (feat. PJ & Rome Fortune)',
 'uts': 1794543092}

## Fetch all pages
If `scrobbles.parquet` already exists, only scrobbles since the newest saved one are fetched. Set `FULL_REFRESH = True` to pull the whole history again. The number of pages comes from the API response. Set `TEST_PAGES` to a small number for a test run, or leave it as `None` to fetch everything. Failed requests are retried, with a short pause between requests to stay within the API's rate limits.

In [9]:
TEST_PAGES = None  # number of pages for a test run (e.g. 3); None pulls all pages
FULL_REFRESH = False  # True ignores scrobbles.parquet and pulls the whole history again

params.pop('from', None)
saved = None
if os.path.exists('scrobbles.parquet') and not FULL_REFRESH:
    saved = pd.read_parquet('scrobbles.parquet')
    params['from'] = int(saved['played_at'].max().timestamp())  # only ask for plays since the newest saved one
    print(f'{len(saved):,} scrobbles saved, fetching newer ones')

first_page = requests.get(URL, params=params, timeout=30).json()  # the page count for this request
total_pages = int(first_page['recenttracks']['@attr']['totalPages'])
pages_to_fetch = range(1, (TEST_PAGES or total_pages) + 1)

all_rows = []
for page in pages_to_fetch:
    for attempt in range(3):  # retry a failed page up to 3 times
        try:
            all_rows.extend(fetch_page(page))
            break  # success, move on to the next page
        except (requests.RequestException, KeyError) as e:  # KeyError: API error body with no 'recenttracks'
            print(f'page {page}, attempt {attempt + 1} failed: {e}')
            time.sleep(2)
    else:  # runs only if all 3 attempts failed
        raise RuntimeError(f'page {page} failed 3 times; stopping')
    time.sleep(0.25)  # be polite to the API

print(f'{len(all_rows)} rows from {len(pages_to_fetch)} pages')

34,011 scrobbles saved, fetching newer ones
26 rows from 1 pages


## Build the DataFrame
One row per scrobble. The Unix timestamp (`uts`, UTC) is converted to a local-time `played_at` column. Set `TIMEZONE` in `lastfm.env` to your own time zone (UTC if unset).

In [10]:
track_df = pd.DataFrame(all_rows, columns=['artist', 'album', 'track', 'uts'])  # columns stay even with no new scrobbles
print(track_df.shape)

(26, 4)


In [11]:
TIMEZONE = os.environ.get('TIMEZONE', 'UTC')  # IANA time zone name, e.g. 'America/New_York'

track_df['played_at'] = pd.to_datetime(track_df['uts'], unit='s', utc=True).dt.tz_convert(TIMEZONE)

## Clean and save
When there is a saved file, the new scrobbles are added to it first. Duplicate rows are dropped (pages can overlap if new scrobbles come in during the download), the rows are sorted oldest to newest, and the result is saved to Parquet. The file is only written after a full pull. Artists that Last.fm splits into several entries (for example Madvillain, Viktor Vaughn and King Geedorah, which are all MF DOOM) are renamed to one name in `ARTIST_ALIASES`, so their plays count together.

In [12]:
# Last.fm lists these as separate artists; add your own
ARTIST_ALIASES = {name: 'MF DOOM' for name in [
    'MF Doom', 'Madvillain', 'Viktor Vaughn', 'Viktor Vaughan', 'King Geedorah',
    'Metal Fingers', 'Zev Love X', 'DANGERDOOM', 'Dangerdoom',
]}

if saved is not None:  # add the new scrobbles to the saved ones
    track_df = pd.concat([saved[['artist', 'album', 'track', 'uts']], track_df[['artist', 'album', 'track', 'uts']]])
    track_df['played_at'] = pd.to_datetime(track_df['uts'], unit='s', utc=True).dt.tz_convert(TIMEZONE)  # recomputed so a changed TIMEZONE applies to everything

track_df['artist'] = track_df['artist'].replace(ARTIST_ALIASES)  # merged under one name, including plays saved earlier

print('duplicates:', track_df.duplicated().sum())
track_df = (track_df
            .drop_duplicates()
            .sort_values('played_at')
            .reset_index(drop=True))
print(track_df.shape)
if len(pages_to_fetch) == total_pages:  # only overwrite the saved file after a full pull
    track_df.to_parquet('scrobbles.parquet', index=False)
else:
    print('test run, not saving')

duplicates: 1
(34036, 5)


In [13]:
track_df.head()

,artist,album,track,uts,played_at
0,Public Enemy,It Takes a Nation of Millions to Hold Us Back,She Watch Channel Zero?!,1740504880,2025-02-25 10:34:40-07:00
1,Creedence Clearwater Revival,Cosmo's Factory,Run Through the Jungle,1740505697,2025-02-25 10:48:17-07:00
2,MF DOOM,Madvillainy,America's Most Blunted (feat. Quasimoto),1740505945,2025-02-25 10:52:25-07:00
3,The Decemberists,The Crane Wife (10th Anniversary Edition),Shankill Butchers,1740506232,2025-02-25 10:57:12-07:00
4,Aaliyah,I Care 4 U,Are You That Somebody,1740507492,2025-02-25 11:18:12-07:00


## Artist metadata
Top tags and global stats for each artist. Raw API responses are cached as JSON lines, so an interrupted pull can resume and the parsing can be re-run without new API calls.

## Artist API helpers
`call_api` retries temporary errors such as rate limits. `fetch_artist` gets an artist's tags and stats, without the bio text.

In [14]:
CACHE_FILE = 'cache/artist_metadata.jsonl'
os.makedirs('cache', exist_ok=True)

RETRYABLE_ERRORS = {8, 11, 16, 29}  # temporary API errors: operation failed, service offline, try again, rate limit


def call_api(method, **extra_params):
    '''Call a Last.fm API method and return the parsed JSON, retrying temporary failures.'''
    request_params = {'method': method, 'api_key': API_KEY, 'format': 'json', **extra_params}
    for attempt in range(3):
        try:
            result = requests.get(URL, params=request_params, timeout=30).json()
            if result.get('error') in RETRYABLE_ERRORS:
                raise RuntimeError(result.get('message', 'temporary API error'))
            time.sleep(0.25)  # stay well under the API rate limit
            return result  # permanent errors (e.g. artist not found) are returned as-is
        except (requests.RequestException, ValueError, RuntimeError) as e:
            print(f'{method} attempt {attempt + 1} failed: {e}')
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f'{method} failed 3 times')


def fetch_artist(name):
    '''Fetch top tags and global stats for one artist.'''
    top_tags = call_api('artist.getTopTags', artist=name, autocorrect=1)
    info = call_api('artist.getInfo', artist=name, autocorrect=1)
    if 'artist' in info:
        info['artist'].pop('bio', None)  # long text that isn't used
    return {'artist': name, 'top_tags': top_tags, 'info': info}

In [15]:
sample = fetch_artist('Talking Heads')
print(sample['top_tags']['toptags']['tag'][:3])
print(sample['info']['artist']['stats'])

[{'count': 100, 'name': 'new wave', 'url': 'https://www.last.fm/tag/new+wave'}, {'count': 79, 'name': 'post-punk', 'url': 'https://www.last.fm/tag/post-punk'}, {'count': 26, 'name': '80s', 'url': 'https://www.last.fm/tag/80s'}]
{'listeners': '3361114', 'playcount': '149714039'}


## Fetch all artists
Artists already in the cache are skipped, so the cell can be re-run after an interruption. Set `TEST_ARTISTS` to a small number for a test run, or leave it as `None` to fetch every artist that isn't cached yet. Expect roughly a second or two per artist.

In [16]:
TEST_ARTISTS = None  # number of artists for a test run (e.g. 5); None fetches every uncached artist

artists = sorted(pd.read_parquet('scrobbles.parquet', columns=['artist'])['artist'].unique())

done = set()
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE) as f:
        done = {json.loads(line)['artist'] for line in f}

todo = [name for name in artists if name not in done]
print(f'{len(artists)} artists, {len(done)} cached, {len(todo)} to fetch')

artists_to_fetch = todo[:TEST_ARTISTS]  # None slices to the whole list

with open(CACHE_FILE, 'a') as f:
    for i, name in enumerate(artists_to_fetch, start=1):
        f.write(json.dumps(fetch_artist(name)) + '\n')
        f.flush()  # write to disk immediately so an interruption loses nothing
        if i % 100 == 0:
            print(f'{i}/{len(artists_to_fetch)} fetched')

1793 artists, 1792 cached, 4 to fetch


## Parse and save
Tags go in a long table (one row per artist and tag), and artist stats go in their own table. Both are only written after every artist has been fetched.

In [17]:
def parse_tags(record):
    '''One row per (artist, tag), with the tag weight (0-100).'''
    tags = record['top_tags'].get('toptags', {}).get('tag', [])
    if isinstance(tags, dict):  # a single tag can come back as a bare dict
        tags = [tags]
    return [{'artist': record['artist'], 'tag': t['name'].lower(), 'weight': int(t['count'])} for t in tags]


def parse_stats(record):
    '''Global listener and play counts for one artist (None values if the artist wasn't found).'''
    info = record['info'].get('artist')
    if info is None:
        return {'artist': record['artist'], 'artist_mbid': None, 'listeners': None, 'global_plays': None}
    return {
        'artist': record['artist'],
        'artist_mbid': info.get('mbid') or None,
        'listeners': int(info['stats']['listeners']),
        'global_plays': int(info['stats']['playcount']),
    }

In [18]:
with open(CACHE_FILE) as f:
    records = [json.loads(line) for line in f]
current = set(artists)
records = [r for r in records if r['artist'] in current]  # drops cached artists that were merged away

tags_df = pd.DataFrame([row for record in records for row in parse_tags(record)])
artists_df = pd.DataFrame([parse_stats(record) for record in records])
print(tags_df.shape, artists_df.shape)

if len(records) == len(artists):  # only save once every artist is cached
    tags_df.to_parquet('artist_tags.parquet', index=False)
    artists_df.to_parquet('artists.parquet', index=False)
else:
    print(f'{len(records)} of {len(artists)} artists cached, not saving')
tags_df.head()

(14152, 3) (1793, 4)


,artist,tag,weight
0,"10,000 Maniacs",female vocalists,100
1,"10,000 Maniacs",alternative,93
2,"10,000 Maniacs",rock,40
3,"10,000 Maniacs",pop,37
4,"10,000 Maniacs",80s,17


In [19]:
artists_df.head()

,artist,artist_mbid,listeners,global_plays
0,"10,000 Maniacs",b9a06530-1241-4162-836f-7b8e79deaa58,538602,6525593
1,2Pac,382f1005-e9ab-4684-afd4-0bdae4ee37f2,4330228,153576951
2,A$AP Rocky,25b7b584-d952-4662-a8b9-dd8cdfbfeb64,3639360,396995078
3,Abhi The Nomad,NaN,253316,3915167
4,Abhi The Nomad & Kato On The Track,NaN,94,1540


In [20]:
for name in ['scrobbles', 'artists', 'artist_tags']:
    print(name, pd.read_parquet(f'{name}.parquet').shape)

scrobbles (34036, 5)
artists (1793, 4)
artist_tags (14152, 3)


## Artist countries
Country of origin for each artist, from MusicBrainz. Last.fm returns a MusicBrainz ID (MBID) for most artists, and the lookup uses it. Artists without an MBID are left out. MusicBrainz asks for no more than one request per second, so the lookup takes about a second per artist. Raw responses are cached, like the Last.fm ones.

In [21]:
MB_URL = 'https://musicbrainz.org/ws/2/artist/'
MB_HEADERS = {'User-Agent': 'lastfm-data-viz/1.0 (https://github.com/skumagai3/lastfm_data_demos)'}  # MusicBrainz asks clients to identify themselves
MB_CACHE_FILE = 'cache/musicbrainz.jsonl'


def fetch_country(mbid):
    '''Look up one artist on MusicBrainz and return its country and area codes.'''
    for attempt in range(3):
        try:
            response = requests.get(MB_URL + mbid, params={'fmt': 'json'}, headers=MB_HEADERS, timeout=30)
            time.sleep(1.1)  # MusicBrainz allows about one request per second
            if response.status_code == 404:  # unknown MBID
                return {'mbid': mbid, 'country': None, 'area_codes': None}
            response.raise_for_status()
            artist = response.json()
            area = artist.get('area') or {}
            return {'mbid': mbid, 'country': artist.get('country'), 'area_codes': area.get('iso-3166-1-codes')}
        except (requests.RequestException, ValueError) as e:
            print(f'lookup attempt {attempt + 1} failed: {e}')
            time.sleep(2 * (attempt + 1))
    raise RuntimeError('MusicBrainz lookup failed 3 times')

## Look up all artists
MBIDs already in the cache are skipped, so the cell can be re-run after an interruption. Set `TEST_LOOKUPS` to a small number for a test run, or leave it as `None` to look up every MBID that isn't cached yet.

In [22]:
TEST_LOOKUPS = None  # number of lookups for a test run (e.g. 5); None looks up every uncached MBID

mbids = sorted(pd.read_parquet('artists.parquet')['artist_mbid'].dropna().unique())

done = set()
if os.path.exists(MB_CACHE_FILE):
    with open(MB_CACHE_FILE) as f:
        done = {json.loads(line)['mbid'] for line in f}

todo = [mbid for mbid in mbids if mbid not in done]
print(f'{len(mbids)} artists with an MBID, {len(done)} cached, {len(todo)} to look up')

lookups_to_run = todo[:TEST_LOOKUPS]  # None slices to the whole list

with open(MB_CACHE_FILE, 'a') as f:
    for i, mbid in enumerate(lookups_to_run, start=1):
        f.write(json.dumps(fetch_country(mbid)) + '\n')
        f.flush()  # write to disk immediately so an interruption loses nothing
        if i % 100 == 0:
            print(f'{i}/{len(lookups_to_run)} looked up')

1281 artists with an MBID, 1284 cached, 0 to look up


## Parse and save countries
MusicBrainz sets a country for most artists. When it doesn't, the country of the artist's area is used instead. Codes starting with X are placeholders (worldwide, Europe) and are dropped. A few artists that MusicBrainz matched to the wrong act are corrected by hand in `COUNTRY_OVERRIDES`. The table is only written after every MBID has been looked up.

In [23]:
COUNTRY_OVERRIDES = {'TOPS': 'CA', 'Blood Orange': 'GB'}  # artists MusicBrainz matched to the wrong act; add your own


def pick_country(record):
    '''Country code for one lookup: the artist's own country, else its area's country.'''
    code = record['country'] or (record['area_codes'] or [None])[0]
    if code is None or code.startswith('X'):  # X* codes are placeholders such as worldwide or Europe
        return None
    return code


with open(MB_CACHE_FILE) as f:
    lookups = [json.loads(line) for line in f]
country_by_mbid = {record['mbid']: pick_country(record) for record in lookups}

artists_df = pd.read_parquet('artists.parquet')
countries_df = artists_df[['artist', 'artist_mbid']].dropna()
countries_df = (countries_df
                .assign(country=countries_df['artist_mbid'].map(country_by_mbid))
                .dropna(subset=['country'])[['artist', 'country']])
fixes = pd.DataFrame({'artist': list(COUNTRY_OVERRIDES), 'country': list(COUNTRY_OVERRIDES.values())})
fixes = fixes[fixes['artist'].isin(artists_df['artist'])]  # only artists that are in this library
if len(fixes):
    countries_df = pd.concat([countries_df[~countries_df['artist'].isin(fixes['artist'])], fixes], ignore_index=True)
print(f'{len(countries_df)} of {len(artists_df)} artists have a country')

if set(mbids) <= set(country_by_mbid):  # only save once every MBID is cached
    countries_df.to_parquet('artist_countries.parquet', index=False)
else:
    print(f'{len(country_by_mbid)} of {len(mbids)} MBIDs cached, not saving')
countries_df.head()

1205 of 1793 artists have a country


,artist,country
0,"10,000 Maniacs",US
1,2Pac,US
2,A$AP Rocky,US
3,Alex Turner,GB
4,Alexander,US


## Profile
The username and profile picture are saved so the charts can show them in the top-right corner. The picture is the largest size Last.fm has for the account.

In [24]:
import io
from PIL import Image

info_params = {'method': 'user.getinfo', 'user': USER, 'api_key': API_KEY, 'format': 'json'}
user_info = requests.get(URL, params=info_params, timeout=30).json()['user']

with open('profile.json', 'w') as f:
    json.dump({'username': user_info['name']}, f)

image_url = next((size['#text'] for size in reversed(user_info['image']) if size['#text']), None)  # sizes are listed small to large
if image_url:
    picture = requests.get(image_url, timeout=30)
    picture.raise_for_status()
    Image.open(io.BytesIO(picture.content)).convert('RGB').save('profile.png')
print(user_info['name'], image_url)

sk3141 https://lastfm-img.freetls.fastly.net/i/u/300x300/053ac77dbaf3ee95fe4550ba24ba5716.png


## Global chart
The most-played artists on Last.fm overall, with their listener and play counts. `07_mainstream.ipynb` uses it as the baseline for how popular my artists are. Each page is 100 artists, so the default pulls the top 10,000. It is a snapshot, so the cell skips the pull when `global_chart.parquet` already exists. Set `REFRESH_CHART = True` to pull a fresh one.

In [25]:
CHART_PAGES = 100  # pages of 100 artists; 100 pages is the global top 10,000
REFRESH_CHART = False  # True pulls the chart again even if global_chart.parquet exists

if os.path.exists('global_chart.parquet') and not REFRESH_CHART:
    chart_df = pd.read_parquet('global_chart.parquet')
    print(f'{len(chart_df):,} artists saved, skipping the pull')
else:
    chart_rows = []
    for page in range(1, CHART_PAGES + 1):
        result = call_api('chart.gettopartists', limit=100, page=page)
        page_artists = result['artists']['artist']
        if not page_artists:  # ran past the end of the chart
            break
        chart_rows.extend({'artist': a['name'], 'listeners': int(a['listeners']), 'global_plays': int(a['playcount'])}
                          for a in page_artists)
        if page % 20 == 0:
            print(f'{page}/{CHART_PAGES} pages')

    chart_df = pd.DataFrame(chart_rows).drop_duplicates('artist')
    chart_df.to_parquet('global_chart.parquet', index=False)
print(chart_df.shape)
chart_df.head()

10,000 artists saved, skipping the pull
(10000, 3)


,artist,listeners,global_plays
0,Olivia Rodrigo,3429358,721384463
1,Radiohead,8509621,1434676993
2,Ariana Grande,4750635,1241956598
3,The Weeknd,5592866,1185122125
4,Lady Gaga,7992861,1075963449


## Sample data
A slice of the most recent scrobbles, with the matching artist, tag, and country rows, so the other notebooks can run without an API key. The slice is contiguous rather than random, so patterns by time of day stay intact.

In [26]:
SAMPLE_DAYS = 120  # most recent days of scrobbles to include in the shared sample

full = pd.read_parquet('scrobbles.parquet')
cutoff = full['played_at'].max() - pd.Timedelta(days=SAMPLE_DAYS)
sample = full[full['played_at'] > cutoff]

sample_artists = set(sample['artist'])
artists = pd.read_parquet('artists.parquet')
tags = pd.read_parquet('artist_tags.parquet')
countries = pd.read_parquet('artist_countries.parquet')

os.makedirs('sample_data', exist_ok=True)
sample.to_parquet('sample_data/scrobbles.parquet', index=False)
artists[artists['artist'].isin(sample_artists)].to_parquet('sample_data/artists.parquet', index=False)
tags[tags['artist'].isin(sample_artists)].to_parquet('sample_data/artist_tags.parquet', index=False)
countries[countries['artist'].isin(sample_artists)].to_parquet('sample_data/artist_countries.parquet', index=False)
pd.read_parquet('global_chart.parquet').to_parquet('sample_data/global_chart.parquet', index=False)  # the whole chart, it is small
print(f'{len(sample):,} scrobbles, {len(sample_artists):,} artists')

3,897 scrobbles, 907 artists
